# 09 — Paper Figures

**Project:** MARS — Multi-Agent Recommender System for Personalized Learning  
**Purpose:** Generate all 8 publication-quality figures for the paper.

| # | Figure | Type |
|---|--------|------|
| 1 | Cold-start accuracy vs interactions | line chart |
| 2 | Thompson Sampling strategy weights | line chart |
| 3 | Confusion matrix XGBoost 6-class | heatmap |
| 4 | t-SNE student clusters | scatter plot |
| 5 | SHAP values XGBoost | beeswarm |
| 6 | NDCG@K for K=1..20 | line chart |
| 7 | LSTM training curves | loss plot |
| 8 | Knowledge Graph subgraph | network plot |

In [ ]:
import sys, os
sys.path.insert(0, "..")
os.chdir("..")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
from pathlib import Path
from collections import defaultdict

plt.style.use("seaborn-v0_8-paper")
plt.rcParams.update({
    "figure.dpi": 300, "savefig.dpi": 300,
    "font.size": 12, "axes.titlesize": 13,
    "axes.labelsize": 12,
    "savefig.bbox": "tight",
})

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Use tab10 palette; MARS always blue (first colour)
COLORS = plt.cm.tab10.colors
MARS_COLOR = COLORS[0]  # blue

import logging
logging.basicConfig(level=logging.WARNING)

NUM_TAGS = 293
print("Libraries loaded.")

In [ ]:
# ── Load data once ──
from data.loader import EdNetLoader
from data.preprocessor import EdNetPreprocessor

loader = EdNetLoader(data_dir="data/raw")
interactions = loader.load_interactions(sample_users=1000)
questions = loader.questions
lectures = loader.lectures

preprocessor = EdNetPreprocessor(output_dir="data/processed", splits_dir="data/splits")
df_clean = preprocessor.clean(interactions)
df_feat = preprocessor.engineer_features(df_clean)
splits = preprocessor.chronological_split(df_feat)
train_df, val_df, test_df = splits["train"], splits["val"], splits["test"]

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

In [ ]:
def save_fig(fig, name):
    """Save as both PNG and PDF."""
    fig.savefig(RESULTS_DIR / f"{name}.png")
    fig.savefig(RESULTS_DIR / f"{name}.pdf")
    print(f"Saved: {name}.png + .pdf")

---
## Figure 1: Cold-Start — Accuracy vs Number of Interactions

In [ ]:
from agents.prediction_agent import PredictionAgent

# Train MARS prediction agent
pred_agent = PredictionAgent()
pred_agent.train(train_df, epochs=10, batch_size=256, patience=3)

# Evaluate at different context sizes
context_sizes = [2, 5, 10, 15, 20, 30, 40, 50]

def parse_tags(t):
    if isinstance(t, list): return [int(x) for x in t]
    if isinstance(t, str): return [int(x.strip()) for x in t.replace(";", ",").split(",") if x.strip().isdigit()]
    return []

mars_acc, popular_acc, random_acc = [], [], []

# Pre-compute popular tags
tag_fail_count = np.zeros(NUM_TAGS)
for _, row in train_df.iterrows():
    if not row["correct"]:
        for t in parse_tags(row.get("tags", [])):
            if 0 <= t < NUM_TAGS: tag_fail_count[t] += 1
popular_ranked = np.argsort(tag_fail_count)[::-1].tolist()

for cs in context_sizes:
    m_hits, p_hits, r_hits, n_total = 0, 0, 0, 0
    rng = np.random.RandomState(42)

    for uid, grp in test_df.groupby("user_id"):
        grp = grp.sort_values("timestamp")
        if len(grp) < cs + 5:
            continue
        context = grp.iloc[:cs]
        future = grp.iloc[cs:cs+10]

        gt_failed = set()
        for _, row in future.iterrows():
            if not row["correct"]:
                gt_failed.update(parse_tags(row.get("tags", [])))
        if not gt_failed:
            continue

        # MARS
        result = pred_agent.predict_gaps(str(uid), recent=context, threshold=0.0)
        mars_top5 = set(np.argsort(result["gap_probabilities"])[-5:])
        m_hits += int(len(mars_top5 & gt_failed) > 0)

        # Popular
        p_hits += int(len(set(popular_ranked[:5]) & gt_failed) > 0)

        # Random
        r_hits += int(len(set(rng.choice(NUM_TAGS, 5, replace=False)) & gt_failed) > 0)

        n_total += 1

    mars_acc.append(m_hits / max(n_total, 1))
    popular_acc.append(p_hits / max(n_total, 1))
    random_acc.append(r_hits / max(n_total, 1))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(context_sizes, mars_acc, "o-", color=MARS_COLOR, linewidth=2,
        markersize=7, label="MARS (ours)", zorder=3)
ax.plot(context_sizes, popular_acc, "s--", color=COLORS[1], linewidth=1.5,
        markersize=5, label="Popular")
ax.plot(context_sizes, random_acc, "^:", color=COLORS[7], linewidth=1.5,
        markersize=5, label="Random")

ax.set_xlabel("Number of Context Interactions")
ax.set_ylabel("Accuracy@5")
ax.set_title("Figure 1: Cold-Start Performance")
ax.legend()
ax.set_ylim(0, 1.05)
sns.despine()

save_fig(fig, "fig1_cold_start")
plt.show()

---
## Figure 2: Thompson Sampling — Strategy Weight Dynamics

In [ ]:
from agents.recommendation_agent import RecommendationAgent, DEFAULT_STRATEGY_PRIORS
import copy

# Simulate TS dynamics for a user over 200 interactions
np.random.seed(42)
n_steps = 200

# Start with default priors
params = copy.deepcopy(DEFAULT_STRATEGY_PRIORS)
history = {s: [] for s in params}

# Simulate: KB is best early, then CF catches up
for step in range(n_steps):
    # Record weights
    for s, ab in params.items():
        history[s].append(ab["alpha"] / (ab["alpha"] + ab["beta"]))

    # Select strategy
    samples = {s: np.random.beta(ab["alpha"], ab["beta"]) for s, ab in params.items()}
    if step < 20:
        samples.pop("collaborative", None)  # CF not eligible early
    chosen = max(samples, key=samples.get)

    # Simulate reward
    if chosen == "knowledge_based":
        reward = 1.0 if np.random.random() < max(0.7 - step * 0.002, 0.4) else 0.0
    elif chosen == "content_based":
        reward = 1.0 if np.random.random() < 0.5 else 0.0
    else:  # collaborative
        reward = 1.0 if np.random.random() < min(0.3 + step * 0.003, 0.7) else 0.0

    params[chosen]["alpha"] += reward
    params[chosen]["beta"] += (1 - reward)

fig, ax = plt.subplots(figsize=(8, 5))
strategy_colors = {"knowledge_based": COLORS[0], "content_based": COLORS[2],
                   "collaborative": COLORS[1]}
strategy_labels = {"knowledge_based": "Knowledge-Based", "content_based": "Content-Based",
                   "collaborative": "Collaborative"}

for s in ["knowledge_based", "collaborative", "content_based"]:
    ax.plot(range(n_steps), history[s], color=strategy_colors[s],
            linewidth=2, label=strategy_labels[s])

ax.axvline(20, color="gray", linestyle=":", alpha=0.5, label="CF eligible")
ax.set_xlabel("Interaction Number")
ax.set_ylabel("Strategy Weight (E[Beta])")
ax.set_title("Figure 2: Thompson Sampling Strategy Dynamics")
ax.legend(loc="center right")
ax.set_ylim(0, 1.0)
sns.despine()

save_fig(fig, "fig2_thompson_sampling")
plt.show()

---
## Figure 3: Confusion Matrix — XGBoost 6-Class Confidence

In [ ]:
from agents.confidence_agent import ConfidenceAgent, CLASS_NAMES
from agents.diagnostic_agent import DiagnosticAgent
from sklearn.metrics import confusion_matrix

diag = DiagnosticAgent()
irt_params = diag.calibrate_from_interactions(train_df, min_answers_per_q=5)

conf_agent = ConfidenceAgent()
conf_agent.train(train_df, irt_params=irt_params)

# Predict on test
test_clean = test_df.dropna(subset=["elapsed_time", "correct", "changed_answer"]).copy()
y_true = conf_agent._assign_labels(test_clean).values
X_test = conf_agent._build_features(test_clean)
y_pred = conf_agent.model.predict(X_test.values.astype(np.float32))

cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={"label": "Proportion"})
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Figure 3: Normalised Confusion Matrix (6-Class Confidence)")
ax.tick_params(axis="x", rotation=35)
ax.tick_params(axis="y", rotation=0)

save_fig(fig, "fig3_confusion_matrix")
plt.show()

---
## Figure 4: t-SNE Student Clusters

In [ ]:
from agents.personalization_agent import (
    PersonalizationAgent, extract_user_features, FEATURE_NAMES,
)
from sklearn.manifold import TSNE

user_features = extract_user_features(interactions)
pers_agent = PersonalizationAgent()
optimal_k = pers_agent.train_clusters(user_features)

X_scaled = pers_agent.scaler.transform(
    np.nan_to_num(user_features[FEATURE_NAMES].values, nan=0.0)
)
labels = pers_agent.model.predict(X_scaled)

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_2d = tsne.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 7))
for cid in range(optimal_k):
    mask = labels == cid
    name = pers_agent._cluster_names.get(cid, f"Cluster {cid}")
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
              c=[COLORS[cid % len(COLORS)]], s=20, alpha=0.6,
              label=f"{name} (n={mask.sum()})")

ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.set_title("Figure 4: Student Clusters (t-SNE)")
ax.legend(markerscale=2, fontsize=10)
sns.despine()

save_fig(fig, "fig4_tsne_clusters")
plt.show()

---
## Figure 5: SHAP Values — XGBoost Confidence Classifier

In [ ]:
import shap
from agents.confidence_agent import FEATURE_NAMES as CONF_FEATURES

X_sample = X_test.sample(min(1500, len(X_test)), random_state=42)
explainer = shap.TreeExplainer(conf_agent.model)
shap_values = explainer.shap_values(X_sample.values.astype(np.float32))

# Average |SHAP| across classes for beeswarm
if isinstance(shap_values, list):
    mean_shap = np.mean([np.abs(sv) for sv in shap_values], axis=0)
else:
    mean_shap = np.mean(np.abs(shap_values), axis=2)

fig, ax = plt.subplots(figsize=(8, 6))
feat_imp = pd.Series(mean_shap.mean(axis=0), index=CONF_FEATURES).sort_values()
feat_imp.plot.barh(ax=ax, color=MARS_COLOR, edgecolor="black", linewidth=0.3)
ax.set_xlabel("Mean |SHAP value| (avg across 6 classes)")
ax.set_title("Figure 5: SHAP Feature Importance (XGBoost Confidence)")
sns.despine()

save_fig(fig, "fig5_shap_importance")
plt.show()

# Also save beeswarm for class 0 (SOLID)
fig_bee = plt.figure(figsize=(10, 6))
if isinstance(shap_values, list):
    shap.summary_plot(shap_values[0], X_sample.values, feature_names=CONF_FEATURES,
                      show=False, max_display=17)
plt.title("SHAP Beeswarm (Class: SOLID)")
plt.tight_layout()
save_fig(plt.gcf(), "fig5b_shap_beeswarm")
plt.show()

---
## Figure 6: NDCG@K for K=1..20 (All Methods)

In [ ]:
from sklearn.metrics import ndcg_score

# Re-build eval pairs from 08
def build_eval_pairs(tdf, ratio=0.5, min_f=3):
    pairs = []
    for uid, grp in tdf.groupby("user_id"):
        grp = grp.sort_values("timestamp")
        si = max(1, int(len(grp) * ratio))
        ctx, fut = grp.iloc[:si], grp.iloc[si:]
        if len(fut) < min_f: continue
        gt = set()
        for _, r in fut.iterrows():
            if not r["correct"]:
                gt.update(parse_tags(r.get("tags", [])))
        if gt:
            pairs.append((str(uid), ctx, gt))
    return pairs

ep = build_eval_pairs(test_df)
K_values = list(range(1, 21))

# Methods to plot
methods = {}

# MARS
mars_scores_per_user = []
mars_gt = []
for uid, ctx, gt in ep:
    result = pred_agent.predict_gaps(uid, recent=ctx, threshold=0.0)
    mars_scores_per_user.append(np.array(result["gap_probabilities"]))
    mars_gt.append(gt)
methods["MARS (ours)"] = mars_scores_per_user

# Popular baseline
tag_scores_pop = tag_fail_count / (tag_fail_count.max() + 1e-10)
methods["Popular"] = [tag_scores_pop] * len(ep)

# Random
rng = np.random.RandomState(42)
methods["Random"] = [rng.random(NUM_TAGS) for _ in ep]

# Compute NDCG@K for each method
fig, ax = plt.subplots(figsize=(8, 5))
method_colors = {"MARS (ours)": MARS_COLOR, "Popular": COLORS[1], "Random": COLORS[7]}
method_styles = {"MARS (ours)": "o-", "Popular": "s--", "Random": "^:"}

for name, scores_list in methods.items():
    ndcg_at_k = []
    for k in K_values:
        vals = []
        for i, (uid, ctx, gt) in enumerate(ep):
            rel = np.array([1.0 if t in gt else 0.0 for t in range(NUM_TAGS)])
            if rel.sum() > 0:
                vals.append(ndcg_score([rel], [scores_list[i]], k=k))
        ndcg_at_k.append(np.mean(vals) if vals else 0.0)

    ax.plot(K_values, ndcg_at_k, method_styles[name],
            color=method_colors[name], linewidth=2, markersize=5,
            label=name)

ax.set_xlabel("K")
ax.set_ylabel("NDCG@K")
ax.set_title("Figure 6: NDCG@K for K=1..20")
ax.legend()
ax.set_xlim(1, 20)
ax.set_ylim(0, 1.0)
ax.set_xticks([1, 5, 10, 15, 20])
sns.despine()

save_fig(fig, "fig6_ndcg_at_k")
plt.show()

---
## Figure 7: LSTM Training Curves

In [ ]:
# Re-train with full epochs to get smooth curves
from agents.prediction_agent import PredictionAgent as PA2

lstm_agent = PA2()
metrics = lstm_agent.train(train_df, val_df=val_df, epochs=30, batch_size=256, patience=7)

history = metrics["history"]
epochs_range = range(1, len(history["train_loss"]) + 1)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(epochs_range, history["train_loss"], "o-", color=MARS_COLOR,
        markersize=4, linewidth=1.5, label="Train Loss")
ax.plot(epochs_range, history["val_loss"], "s-", color=COLORS[3],
        markersize=4, linewidth=1.5, label="Validation Loss")
ax.axvline(metrics["best_epoch"], color="gray", linestyle="--", alpha=0.6,
           label=f"Best epoch ({metrics['best_epoch']})")

ax.set_xlabel("Epoch")
ax.set_ylabel("BCE Loss")
ax.set_title("Figure 7: LSTM Gap Prediction — Training Curves")
ax.legend()
sns.despine()

save_fig(fig, "fig7_lstm_training")
plt.show()

print(f"Best epoch: {metrics['best_epoch']}, Val loss: {metrics['val_loss']}, Val AUC: {metrics['val_auc']}")

---
## Figure 8: Knowledge Graph — Top-20 Tag Dependency Subgraph

In [ ]:
from agents.kg_agent import KnowledgeGraphAgent, PART_NAMES

kg_agent = KnowledgeGraphAgent()
kg_agent.build_graph(questions, lectures)
kg_agent.build_prerequisites(train_df)
kg_agent.update_difficulties(train_df)

stats = kg_agent.get_graph_stats()
print(f"Graph: {stats['total_nodes']} nodes, {stats['total_edges']} edges")
print(f"Prerequisites: {stats['prerequisite_dag_edges']} edges, DAG: {stats['prerequisite_is_dag']}")

In [ ]:
# Extract top-20 most-connected tags and their prerequisite edges
G = kg_agent.graph

# Get all tag nodes
tag_nodes = [
    (n, d) for n, d in G.nodes(data=True)
    if d.get("node_type") == "tag"
]

# Sort by degree (most connected)
tag_degrees = [(n, G.degree(n), d) for n, d in tag_nodes]
tag_degrees.sort(key=lambda x: -x[1])
top20_tags = {x[0] for x in tag_degrees[:20]}

# Build subgraph with prereq edges + BELONGS_TO_PART edges for context
sub = nx.DiGraph()

for n in top20_tags:
    d = G.nodes[n]
    tid = d.get("tag_id", n)
    diff = d.get("avg_difficulty", 0.5)
    qcount = d.get("question_count", 0)
    sub.add_node(n, label=f"Tag {tid}", difficulty=diff,
                 size=max(qcount, 5), node_type="tag")

# Add prereq edges between top-20 tags
for u, v, d in G.edges(data=True):
    if d.get("edge_type") == "PREREQUISITE_OF" and u in top20_tags and v in top20_tags:
        sub.add_edge(u, v, weight=d.get("strength", 0.5))

# Add part nodes connected to these tags
for tn in top20_tags:
    parts = G.nodes[tn].get("part_ids", set())
    if isinstance(parts, set):
        for pid in list(parts)[:1]:  # primary part only
            pn = f"part_{pid}"
            if not sub.has_node(pn):
                sub.add_node(pn, label=f"Part {pid}", node_type="part", size=30)
            sub.add_edge(tn, pn)

# Draw
fig, ax = plt.subplots(figsize=(12, 10))

pos = nx.spring_layout(sub, k=2.5, seed=42, iterations=50)

# Node colours and sizes
node_colors = []
node_sizes = []
for n in sub.nodes():
    d = sub.nodes[n]
    if d.get("node_type") == "part":
        node_colors.append("#e74c3c")
        node_sizes.append(500)
    else:
        diff = d.get("difficulty", 0.5)
        node_colors.append(plt.cm.YlOrRd(diff))
        node_sizes.append(max(d.get("size", 10) * 3, 80))

# Draw edges
prereq_edges = [(u, v) for u, v in sub.edges() if sub.nodes[v].get("node_type") != "part"]
part_edges = [(u, v) for u, v in sub.edges() if sub.nodes[v].get("node_type") == "part"]

nx.draw_networkx_edges(sub, pos, edgelist=prereq_edges, ax=ax,
                       edge_color="#2c3e50", arrows=True,
                       arrowsize=15, width=1.5, alpha=0.7,
                       connectionstyle="arc3,rad=0.1")
nx.draw_networkx_edges(sub, pos, edgelist=part_edges, ax=ax,
                       edge_color="#bdc3c7", arrows=False,
                       width=0.8, alpha=0.5, style="dashed")

# Draw nodes
nx.draw_networkx_nodes(sub, pos, ax=ax, node_color=node_colors,
                       node_size=node_sizes, edgecolors="black",
                       linewidths=0.5)

# Labels
labels = {n: sub.nodes[n].get("label", n) for n in sub.nodes()}
nx.draw_networkx_labels(sub, pos, labels, ax=ax, font_size=8, font_weight="bold")

# Legend
legend_elements = [
    mpatches.Patch(facecolor="#e74c3c", label="TOEIC Part"),
    mpatches.Patch(facecolor="#f4d03f", label="Tag (colour = difficulty)"),
    plt.Line2D([0], [0], color="#2c3e50", linewidth=2, label="Prerequisite"),
    plt.Line2D([0], [0], color="#bdc3c7", linewidth=1, linestyle="--", label="Belongs to Part"),
]
ax.legend(handles=legend_elements, loc="upper left", fontsize=10)

ax.set_title("Figure 8: Knowledge Graph — Top-20 Tag Dependencies")
ax.axis("off")

save_fig(fig, "fig8_knowledge_graph")
plt.show()

print(f"Subgraph: {sub.number_of_nodes()} nodes, {sub.number_of_edges()} edges")

---
## Summary

In [ ]:
import glob

figs = sorted(glob.glob(str(RESULTS_DIR / "fig*.png")))
print("\n" + "=" * 50)
print("  ALL 8 FIGURES GENERATED")
print("=" * 50)
for f in figs:
    print(f"  {Path(f).name}")
print(f"\n  Total: {len(figs)} PNG + {len(figs)} PDF files")
print(f"  Directory: {RESULTS_DIR.resolve()}")
print("=" * 50)